# 🏷️ Lesson 3.2 — Classification: Teaching AI to Sort Things

**AI Crash Course for Absolute Beginners**

Classification is the most common ML task. In this notebook:
- Build a Logistic Regression classifier from scratch intuition
- Train Decision Tree and K-Nearest Neighbours models
- Evaluate with accuracy, precision, recall, and confusion matrix
- Visualise decision boundaries

> Run each cell with **Shift + Enter**.

In [ ]:
# Install the libraries we need (Colab usually has these, but this ensures they are up to date)
!pip install numpy pandas matplotlib scikit-learn --quiet

# numpy  — fast number crunching (arrays, maths)
import numpy as np

# pandas — working with tables of data (like Excel in Python)
import pandas as pd

# matplotlib — drawing charts and graphs
import matplotlib.pyplot as plt

# scikit-learn — the most popular ML library in Python
# We import specific tools from it:
from sklearn.datasets import load_breast_cancer          # a real medical dataset built into scikit-learn
from sklearn.model_selection import train_test_split     # splits data into training and testing sets
from sklearn.preprocessing import StandardScaler         # scales features to have mean=0, std=1
from sklearn.linear_model import LogisticRegression      # Model 1: logistic regression classifier
from sklearn.tree import DecisionTreeClassifier, plot_tree  # Model 2: decision tree + a tool to draw it
from sklearn.neighbors import KNeighborsClassifier       # Model 3: K-Nearest Neighbours
from sklearn.metrics import (
    accuracy_score,          # percentage of correct predictions
    classification_report,   # full precision/recall/F1 breakdown
    confusion_matrix,        # table showing where the model got confused
    ConfusionMatrixDisplay   # tool to draw the confusion matrix as a chart
)

---
## The Dataset: Breast Cancer Wisconsin
569 tumour measurements. Goal: classify as **malignant** (0) or **benign** (1).

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")   # 0=malignant, 1=benign

print(f"Features : {X.shape[1]}")
print(f"Samples  : {X.shape[0]}")
print(f"Classes  : {dict(zip(data.target_names, np.bincount(y)))}")
X.iloc[:, :5].describe().round(2)

In [ ]:
# --- Split the data into training and testing sets ---
# test_size=0.2 means 20% of data goes to testing, 80% to training
# random_state=42 means the split is the same every time you run it (reproducible)
# stratify=y ensures both splits have the same ratio of malignant to benign (important for imbalanced data)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Scale the features ---
# Many ML models work better when all features are on the same scale
# StandardScaler transforms each feature to have mean=0 and standard deviation=1
scaler = StandardScaler()

# fit_transform on TRAINING data: learns the mean/std from training data, then scales it
X_train_s = scaler.fit_transform(X_train)

# transform on TEST data: uses the SAME mean/std learned from training (never fit on test data!)
# This is important — fitting on test data would be "cheating"
X_test_s  = scaler.transform(X_test)

print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples")

---
## Model 1 — Logistic Regression

Despite the name, Logistic Regression is a **classifier**. It estimates the probability of belonging to a class.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)

y_pred_lr = lr.predict(X_test_s)
print(f"Logistic Regression accuracy: {accuracy_score(y_test, y_pred_lr):.3f}")

# Probability output (not just 0 or 1)
proba = lr.predict_proba(X_test_s)[:5]
print("\nFirst 5 probability predictions:")
print("  P(malignant)  P(benign)  Prediction")
for p, pred in zip(proba, y_pred_lr[:5]):
    label = data.target_names[pred]
    print(f"     {p[0]:.3f}       {p[1]:.3f}      {label}")

---
## Model 2 — Decision Tree

In [ ]:
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train_s, y_train)

y_pred_dt = dt.predict(X_test_s)
print(f"Decision Tree accuracy: {accuracy_score(y_test, y_pred_dt):.3f}")

# Visualise the tree
plt.figure(figsize=(18, 6))
plot_tree(dt, feature_names=data.feature_names, class_names=data.target_names,
          filled=True, rounded=True, fontsize=8)
plt.title("Decision Tree (max_depth=4)")
plt.tight_layout()
plt.show()

---
## Model 3 — K-Nearest Neighbours (KNN)

In [ ]:
# Try different values of K to find which works best
# K is the number of neighbours the model looks at when making a prediction
k_results = []
for k in range(1, 21):   # test K from 1 to 20
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)           # train the model
    acc = accuracy_score(y_test, knn.predict(X_test_s))  # measure accuracy on test data
    k_results.append((k, acc))           # store the result as a (k, accuracy) pair

# zip(*k_results) unpacks the list of pairs into two separate lists: k values and accuracies
k_vals, accs = zip(*k_results)
best_k = k_vals[accs.index(max(accs))]  # find the K with the highest accuracy

# Plot accuracy vs K to visually find the best value
plt.figure(figsize=(8, 3))
plt.plot(k_vals, accs, marker="o", color="#1a6bc8")
plt.axvline(best_k, color="#c8401a", linestyle="--", label=f"Best k={best_k}")
plt.xlabel("Number of Neighbours (k)")
plt.ylabel("Test Accuracy")
plt.title("KNN: Accuracy vs K")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Best k={best_k}, accuracy={max(accs):.3f}")

---
## Evaluation: More Than Just Accuracy

In [ ]:
# For medical diagnosis, false negatives (missed cancer) are very costly
print("=== Logistic Regression — Full Report ===")
print(classification_report(y_test, y_pred_lr, target_names=data.target_names))

# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (model, pred, name) in zip(axes, [
    (lr, y_pred_lr, "Logistic Regression"),
    (dt, y_pred_dt, "Decision Tree")
]):
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=data.target_names).plot(ax=ax, colorbar=False)
    ax.set_title(name)

plt.suptitle("Confusion Matrices", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Model comparison table
from sklearn.metrics import precision_score, recall_score, f1_score

knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_s, y_train)
y_pred_knn = knn_best.predict(X_test_s)

models = {
    "Logistic Regression": y_pred_lr,
    "Decision Tree"      : y_pred_dt,
    f"KNN (k={best_k})"  : y_pred_knn
}

results = []
for name, pred in models.items():
    results.append({
        "Model"    : name,
        "Accuracy" : f"{accuracy_score(y_test, pred):.3f}",
        "Precision": f"{precision_score(y_test, pred):.3f}",
        "Recall"   : f"{recall_score(y_test, pred):.3f}",
        "F1"       : f"{f1_score(y_test, pred):.3f}"
    })

pd.DataFrame(results).set_index("Model")

---
## Challenge Exercises

1. **Try Random Forest**: `from sklearn.ensemble import RandomForestClassifier` — does it beat all three models?
2. **Feature importance**: After fitting a tree, print `dt.feature_importances_` and plot the top 10.
3. **ROC curve**: Plot `from sklearn.metrics import roc_curve` for Logistic Regression and find the AUC.

---
**Next lesson:** [Lesson 3.3 — Regression](https://colab.research.google.com/github/GirlEf/ai-crash-course/blob/main/notebooks/lesson-3.3-regression.ipynb)